# Train one attention head on a toy task
Task: at each position, output the token that came **right before** it. We start with random `Wq`, `Wk`, `Wv` and watch the `T × T` attention weights change.

The model also needs position embeddings. Without them, attention cannot know which token is 'before'.

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt
torch.manual_seed(0)
V, T, d_in, d_out = 10, 8, 16, 16   # vocab, context_size, d_in, d_out

In [ ]:
class Head(nn.Module):
    def __init__(s):
        super().__init__()
        s.tok = nn.Embedding(V, d_in); s.pos = nn.Embedding(T, d_in)
        s.Wq = nn.Linear(d_in, d_out, bias=False)
        s.Wk = nn.Linear(d_in, d_out, bias=False)
        s.Wv = nn.Linear(d_in, d_out, bias=False)
        s.out = nn.Linear(d_out, V)
        s.register_buffer("mask", torch.triu(torch.ones(T, T), 1).bool())
    def forward(s, idx):                       # idx: (B, T)
        x = s.tok(idx) + s.pos(torch.arange(T))  # (B, T, d_in)
        Q, K, Vv = s.Wq(x), s.Wk(x), s.Wv(x)     # (B, T, d_out)
        scores = Q @ K.transpose(1, 2) / d_out**0.5   # (B, T, T)
        scores = scores.masked_fill(s.mask, float("-inf"))
        w = torch.softmax(scores, dim=-1)        # (B, T, T)
        s.last_w = w
        return s.out(w @ Vv)                     # (B, T, V)
m = Head()

In [ ]:
def batch(B=64):
    idx = torch.randint(0, V, (B, T))
    tgt = torch.roll(idx, 1, dims=1)   # token before; position 0 wraps, so we skip it in the loss
    return idx, tgt

def show(title):
    idx, _ = batch(256); m(idx)
    plt.imshow(m.last_w.mean(0).detach(), cmap="viridis", vmin=0, vmax=1)
    plt.title(title); plt.xlabel("attends to (j)"); plt.ylabel("token (i)"); plt.colorbar(); plt.show()

show("Before training: average attention weights")

In [ ]:
opt = torch.optim.Adam(m.parameters(), lr=3e-3)
for step in range(1, 801):
    idx, tgt = batch()
    logits = m(idx)
    loss = nn.functional.cross_entropy(logits[:, 1:].reshape(-1, V), tgt[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward()
    if step in (1, 800):
        print(step, "grad norms  Wq %.4f  Wk %.4f  Wv %.4f" % tuple(p.weight.grad.norm() for p in (m.Wq, m.Wk, m.Wv)))
    opt.step()
    if step % 100 == 0: print(step, "loss", round(loss.item(), 3))

In [ ]:
show("After training: average attention weights")

## Questions to answer
1. Each row should now put its weight on column `i-1`. Why is the upper triangle still 0?
2. The three grad norms differ. Why? (See the earlier note on where each matrix sits in the formula.)
3. Change `d_out` to 4. What breaks, and what does not?